Data Cleaning in Pandas

In [27]:
import pandas as pd

In [29]:
#understand the dataset

df_game_penalties = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\extract\raw_data\game_penalties.csv")
df_game_penalties.head()
df_game_penalties.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247828 entries, 0 to 247827
Data columns (total 3 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   play_id          247828 non-null  object
 1   penaltySeverity  120299 non-null  object
 2   penaltyMinutes   247828 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 5.7+ MB


In [30]:
# Create a new dataframe from df_game_goals (copying original to preserve data)
df_clean_game_penalties = df_game_penalties.copy()

Rename columns 

In [34]:
#Rename columns for consistency
import re

# Function to add an underscore before uppercase letters and convert to lowercase
def rename_columns(col_name):
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col_name).lower()

# Apply the function to all column names
df_clean_game_penalties.columns = [rename_columns(col) for col in df_clean_game_penalties.columns]
df_clean_game_penalties.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247828 entries, 0 to 247827
Data columns (total 3 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   play_id           247828 non-null  object
 1   penalty_severity  120299 non-null  object
 2   penalty_minutes   247828 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 5.7+ MB


Check : Null 

In [37]:
# Null - All - check for missing or null values entire dataset
# Results : game_winning_goal = 1844
#           empty_net = 5043

df_clean_game_penalties.isnull().sum()

play_id                  0
penalty_severity    127529
penalty_minutes          0
dtype: int64

In [38]:
# explore columns with null values
# Action : 'penalty_severity' column leave as null
df_clean_game_penalties[df_clean_game_penalties.isnull().any(axis=1)]

,play_id,penalty_severity,penalty_minutes
76634,2009021012_1,NaN,2
76635,2009021012_2,NaN,2
76636,2009021012_3,NaN,2
76637,2009021012_4,NaN,2
76638,2009021012_5,NaN,2
...,...,...,...
204158,2001020101_7,NaN,2
204159,2001020101_9,NaN,2
204160,2001020101_11,NaN,2
204161,2001020101_12,NaN,2


In [41]:
# Count unique 'game_id' values
unique_ids = df_clean_game_penalties['play_id'].nunique()

# Count total number of rows
total_rows = len(df_clean_game_penalties)

# Display the results
print(f"Unique game_ids: {unique_ids}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_ids} rows with duplicates.")

Unique game_ids: 229228
Total rows: 247828
There are 18600 rows with duplicates.


In [43]:
# Result : 18600 duplicates
# Additional Duplicate Check for 'play_id', 'penalty_severity', 'penalty_minutes'combinations
duplicates_id = df_clean_game_penalties[df_clean_game_penalties.duplicated(subset=['play_id', 'penalty_severity', 'penalty_minutes'], keep=False)]

# Sort rows with duplicated results in ascending order by 'play_id', 'penalty_severity', 'penalty_minutes'
duplicates_id_sorted = duplicates_id.sort_values(by=['play_id', 'penalty_severity', 'penalty_minutes'], ascending=True)

# Count total number of rows with duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes'
total_duplicates = df_clean_game_penalties.duplicated(subset=['play_id', 'penalty_severity', 'penalty_minutes'], keep=False).sum()

# Count the unique combinations of 'play_id', 'penalty_severity', 'penalty_minutes'
unique_game_official_combinations = df_clean_game_penalties[['play_id', 'penalty_severity', 'penalty_minutes']].drop_duplicates().shape[0]

# Count total number of rows
total_rows = len(df_clean_game_penalties)

# Display the results
print(f"Unique 'play_id', 'penalty_severity', 'penalty_minutes' combinations: {unique_game_official_combinations}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_game_official_combinations} rows with duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes'.")

# Display the sorted duplicates for further review
print("Sorted Duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes':")
print(duplicates_id_sorted[['play_id', 'penalty_severity', 'penalty_minutes']])

Unique 'play_id', 'penalty_severity', 'penalty_minutes' combinations: 229228
Total rows: 247828
There are 18600 rows with duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes'.
Sorted Duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes':
               play_id penalty_severity  penalty_minutes
228407  2018020001_127            Minor                2
228443  2018020001_127            Minor                2
228408  2018020001_156            Minor                2
228444  2018020001_156            Minor                2
228409  2018020001_160      Bench Minor                2
...                ...              ...              ...
228405  2019030416_315            Minor                2
228394   2019030416_34            Minor                2
228400   2019030416_34            Minor                2
228395   2019030416_59            Minor                2
228401   2019030416_59            Minor                2

[37200 rows x 3 columns]


In [44]:
# Remove duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes'
df_clean_game_penalties = df_clean_game_penalties.drop_duplicates(subset=['play_id', 'penalty_severity', 'penalty_minutes'], keep='first')

# Display the cleaned dataframe (first 5 rows as an example)
df_clean_game_penalties.head(5)

,play_id,penalty_severity,penalty_minutes
0,2016020045_41,Minor,2
1,2016020045_101,Minor,2
2,2016020045_134,Minor,2
3,2016020045_174,Minor,2
4,2016020045_189,Minor,2


In [45]:
#debugging
try:
    result_index = df_clean_game_penalties[df_clean_game_penalties['play_id'] == '2016020045_41'].index[0]  # Get the index of the match
    result_row = df_clean_game_penalties.iloc[result_index]  # Retrieve the row using iloc
    print("Found the play_id '2016020045_41':")
    print(result_row)
except IndexError:
    print("The play_id '2016020045_41' does not exist in the games_play table.")

Found the play_id '2016020045_41':
play_id             2016020045_41
penalty_severity            Minor
penalty_minutes                 2
Name: 0, dtype: object


In [47]:
# No more duplicates. Resolved!
# Additional Duplicate Check for 'play_id', 'penalty_severity', 'penalty_minutes'combinations
duplicates_id = df_clean_game_penalties[df_clean_game_penalties.duplicated(subset=['play_id', 'penalty_severity', 'penalty_minutes'], keep=False)]

# Sort rows with duplicated results in ascending order by 'play_id', 'penalty_severity', 'penalty_minutes'
duplicates_id_sorted = duplicates_id.sort_values(by=['play_id', 'penalty_severity', 'penalty_minutes'], ascending=True)

# Count total number of rows with duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes'
total_duplicates = df_clean_game_penalties.duplicated(subset=['play_id', 'penalty_severity', 'penalty_minutes'], keep=False).sum()

# Count the unique combinations of 'play_id', 'penalty_severity', 'penalty_minutes'
unique_game_official_combinations = df_clean_game_penalties[['play_id', 'penalty_severity', 'penalty_minutes']].drop_duplicates().shape[0]

# Count total number of rows
total_rows = len(df_clean_game_penalties)

# Display the results
print(f"Unique 'play_id', 'penalty_severity', 'penalty_minutes' combinations: {unique_game_official_combinations}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_game_official_combinations} rows with duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes'.")

# Display the sorted duplicates for further review
print("Sorted Duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes':")
print(duplicates_id_sorted[['play_id', 'penalty_severity', 'penalty_minutes']])

Unique 'play_id', 'penalty_severity', 'penalty_minutes' combinations: 229228
Total rows: 229228
There are 0 rows with duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes'.
Sorted Duplicates based on 'play_id', 'penalty_severity', 'penalty_minutes':
Empty DataFrame
Columns: [play_id, penalty_severity, penalty_minutes]
Index: []


In [50]:
#save file locally
df_clean_game_penalties.to_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_penalties.csv", index=False)